In [ ]:
import sys
import numpy as np

import geopandas as gpd
import rasterio
from rasterio.mask import mask

import time
import calendar

import pystac_client
from pystac_client.stac_api_io import APIError
from rasterio.errors import RasterioIOError
import planetary_computer
from IPython.display import clear_output
from rasterio.env import Env

sys.path.append("../utils")

In [ ]:
inspections = gpd.read_file(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master_training_geometries.geojson"
).drop(columns = 'apn')
inspections.head()


In [ ]:
# Two Months Prior NDVI Function
# Built from Third Iteration Function

# from datetime import datetime
# from dateutil.relativedelta import relativedelta
from calendar import monthrange


def add_mean_ndvi_2month_prior(dataframe):
    # This is to avoid errors from .tifs in URLs
    env = Env(CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.TIF")

    df_out = dataframe.copy()
    df_query = df_out.to_crs(epsg=4326)

    # Open the catalog once
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    # 2) Compute the two-months-prior year & month exactly like your rain example
    df_query["month_prior_2"] = df_query["month"].apply(
        lambda x: x - 2 if x > 2 else x + 10
    )
    df_query["year_prior_2"] = df_query["year"]
    df_query.loc[df_query["month"].isin([1, 2]), "year_prior_2"] = df_query["year"] - 1

    mean_vals = []
    for insp_id in df_query["inspection_id"]:
        row = df_query.loc[df_query["inspection_id"] == insp_id].iloc[0]
        geom = row.geometry

        # 3) Pull your prior-year & prior-month right off the row
        p_year = int(row["year_prior_2"])
        p_month = int(row["month_prior_2"])

        # 4) Build your start/end strings
        start = f"{p_year}-{p_month:02d}-01"
        last_day = monthrange(p_year, p_month)[1]
        end = f"{p_year}-{p_month:02d}-{last_day:02d}"

        # 5) Fire off your STAC query exactly as before
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=geom.bounds,
                datetime=f"{start}/{end}",
                query={"eo:cloud_cover": {"lte": 25}},
            )
            item = next(search.items(), None)
        except APIError:
            mean_vals.append(np.nan)
            continue

        if item is None:
            print(f"inspection_id {insp_id}: no scene found, NaN")
            mean_vals.append(np.nan)
            continue

        # Get CRS of raster
        with rasterio.open(item.assets["B04"].href) as src:
            cog_crs = src.crs

        # Reproject geometry
        single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
        mask_geom = [single.geometry.iloc[0]]

        # Clip and compute NDVI, avoiding errors. If error, input NaN
        try:
            with env:
                with (
                    rasterio.open(item.assets["B04"].href) as src_red,
                    rasterio.open(item.assets["B08"].href) as src_nir,
                ):
                    red_clip, _ = mask(src_red, mask_geom, crop=True)
                    nir_clip, _ = mask(src_nir, mask_geom, crop=True)
        except (ValueError, RasterioIOError) as e:
            print(f"inspection_id {insp_id}: masking error ({e}) → NaN")
            mean_vals.append(np.nan)
            continue

        # Calculate NDVI, suppressing warnings.
        red = red_clip.astype("float32")
        nir = nir_clip.astype("float32")
        with np.errstate(divide="ignore", invalid="ignore"):
            ndvi = (nir - red) / (nir + red)

        mean_val = float(np.nanmean(ndvi))
        mean_vals.append(mean_val)
        print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

    df_out["mean_ndvi_2month_prior"] = mean_vals
    return df_out


In [ ]:
# Dataframe split; running the whole frame takes way too long and causes an API timeout. We'll do it in tenths.

# total rows
n = len(inspections)

# compute a chunk size so that the first 9 are equal and the last picks up any remainder
chunk_size = n // 10
remainder = n % 10

splits = []
start = 0
for i in range(10):
    extra = 1 if i < remainder else 0
    stop = start + chunk_size + extra
    splits.append(inspections.iloc[start:stop])
    start = stop

# now print out the sizes
for i, split_df in enumerate(splits, start=1):
    print(f"Split {i}: {len(split_df)} rows, {len(split_df.columns)} columns")

In [ ]:
# Maybe you already processed some splits! Select the split to start from here.
start_split = 1

for i, split_df in enumerate(splits[start_split - 1 :], start=start_split):
    clear_output(wait=True)
    print(f"--- Processing split {i} of {len(splits)} ---")

    # Compute NDVI for this chunk
    df_chunk = add_mean_ndvi_2month_prior(split_df)

    # Report how many NaNs were produced
    n_missing = df_chunk["mean_ndvi_2month_prior"].isna().sum()
    
    # Count how many mean_ndvi values are below -0.1 (likely clouds)
    n_clouds = (df_chunk["mean_ndvi_2month_prior"] < -0.1).sum()
    print(
        f"Split {i}: {n_missing} missing NDVI values; "
        f"{n_clouds} values below -0.1 (likely clouds)"
    )
    
    # Save only the needed columns
    df_chunk = df_chunk[["inspection_id", "mean_ndvi_2month_prior"]]

    # Save to CSV, embedding the split number in the filename
    out_path = f"/capstone/wildfire_prep/ryan/data-preparation/code/07_ndvi_pystac/ndvi_2month_prior_files/prior_ndvi_{i:02d}_of10.csv"
    df_chunk.to_csv(out_path, index=False)
